In [8]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from datetime import datetime

BASE_URL = "https://www.tutorialsfreak.com/"  # Replace later

HEADERS = {
    'User-Agent': 'Mozilla/5.0'
}

In [10]:
def get_soup(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.content, 'html.parser')
    except Exception as e:
        print(f"Error: {e}")
        return None

In [11]:
def scrape_hospital_page(url):
    soup = get_soup(url)
    if not soup:
        return None

    data = {}

    # Doctor Name
    name = soup.find('h1')
    data['doctor_name'] = name.text.strip() if name else 'N/A'

    # Specialty
    specialty = soup.find('span', class_='specialty')
    data['specialty'] = specialty.text.strip() if specialty else 'General'

    # Hospital Name
    hospital = soup.find('div', class_='hospital-name')
    data['hospital'] = hospital.text.strip() if hospital else 'N/A'

    # Location
    location = soup.find('div', class_='location')
    data['location'] = location.text.strip() if location else 'N/A'

    # Fees (optional)
    fees = soup.find('span', class_='fees')
    data['fees'] = fees.text.strip() if fees else 'N/A'

    # Extra metrics (like your project)
    data['total_links'] = len(soup.find_all('a'))
    data['total_images'] = len(soup.find_all('img'))

    data['scraped_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    return data

In [12]:
def get_doctor_links():
    soup = get_soup(BASE_URL)
    if not soup:
        return []

    links = []

    for a in soup.find_all('a'):
        href = a.get('href')

        if href and href.startswith('http'):
            links.append(href)

    return list(set(links))

In [13]:
def scrape_all_doctors(limit=10):
    all_data = []

    links = get_doctor_links()
    print(f"Found {len(links)} links")

    for i, link in enumerate(links[:limit], 1):
        print(f"\nScraping {i}: {link}")

        data = scrape_hospital_page(link)

        if data:
            data['url'] = link
            all_data.append(data)
            print("✓ Success")
        else:
            print("✗ Failed")

        time.sleep(1)

    return all_data

In [14]:
def save_to_csv(data, filename="hospital_data.csv"):
    headers = [
        'doctor_name', 'specialty', 'hospital',
        'location', 'fees', 'total_links',
        'total_images', 'scraped_at', 'url'
    ]

    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data)

    print(f"Saved to {filename}")

In [15]:
print("Starting Hospital Scraper...")

doctors = scrape_all_doctors(limit=10)

save_to_csv(doctors)

Starting Hospital Scraper...
Found 12 links

Scraping 1: https://www.wscubetech.com/online-python-certification-course-india.html?utm_source=TutorialsFreak&utm_medium=python-tutorial&utm_campaign=leads
✓ Success

Scraping 2: https://www.linkedin.com/company/tutorialsfreak/
✓ Success

Scraping 3: https://bit.ly/seo-course-ws
✓ Success

Scraping 4: https://www.wscubetech.com/online-data-analytics-course.html?utm_source=TutorialsFreak&utm_medium=ai-tutorial&utm_campaign=leads
✓ Success

Scraping 5: https://twitter.com/TutorialsFreak
✓ Success

Scraping 6: https://www.instagram.com/tutorials_freak/
Error: HTTPSConnectionPool(host='www.instagram.com', port=443): Read timed out. (read timeout=10)
✗ Failed

Scraping 7: https://bit.ly/web-developer-course-ws
✓ Success

Scraping 8: https://www.wscubetech.com/
✓ Success

Scraping 9: https://bit.ly/digital-marketing-course-ws
✓ Success

Scraping 10: https://www.wscubetech.com/online-ethical-hacking-course.html?utm_source=TutorialsFreak&utm_medium